# Praktikum Clustering K-Means — Global Earthquakes Seismic Telemetry
**Tujuan:** mengelompokkan kejadian gempa bumi global berdasarkan karakteristik seismik memakai K-Means, mengevaluasi hasilnya (Inertia & Silhouette), dan menafsirkan klaster.

**Dataset:** https://www.kaggle.com/datasets/luthfimuthathohirin/global-earthquakes-seismic-telemetry  
Letakkan file CSV (`global_earthquakes_seismic*.csv`) satu folder dengan notebook ini.

In [ ]:
import glob, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## Langkah 1 — Pemahaman dan Pengambilan Dataset

In [ ]:
path = sorted(glob.glob("global_earthquakes_seismic*.csv"))[0]
df = pd.read_csv(path)
print("Jumlah baris, kolom:", df.shape)
df.info()

In [ ]:
df.head()

In [ ]:
FEATS = ["depth", "mag", "sig", "nst", "dmin", "rms", "gap"]
df[FEATS].describe().T

**Fitur untuk clustering:** `depth` (kedalaman, km), `mag` (magnitudo), `sig` (skor signifikansi), `nst` (jumlah stasiun), `dmin` (jarak ke stasiun terdekat), `rms` (residu waktu tempuh), `gap` (celah azimut).  
`latitude`/`longitude` tidak dipakai karena jarak Euclidean pada koordinat tidak bermakna secara geografis; keduanya dipakai untuk visualisasi peta saja. Kolom teks (`place`, `title`, `evidence_span`, `cot_rationale`) dan kolom turunan kategorikal tidak dipakai sebagai fitur.

## Langkah 2 — Eksplorasi Data (EDA)

In [ ]:
print("Missing value per kolom (total):", int(df.isna().sum().sum()))
print("Baris duplikat:", int(df.duplicated().sum()))
print(df["type"].value_counts())
print("Skewness:\n", df[FEATS].skew().round(2))

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(16, 7))
for a, f in zip(ax.ravel(), FEATS):
    sns.histplot(df[f], bins=40, kde=True, ax=a, color="#2a6f97"); a.set_title(f)
ax.ravel()[-1].axis("off"); plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(16, 7))
for a, f in zip(ax.ravel(), FEATS):
    sns.boxplot(y=df[f], ax=a, color="#a9d6e5"); a.set_title(f)
ax.ravel()[-1].axis("off"); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(7, 6))
sns.heatmap(df[FEATS].corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1)
plt.title("Korelasi antar fitur"); plt.show()

**Catatan EDA:** tidak ada missing value maupun duplikasi. `depth`, `dmin`, `sig`, dan `mag` condong ke kanan (ekor panjang) karena gempa dalam dan gempa besar jarang terjadi. Korelasi terkuat: `mag`–`sig` (≈0,80) dan `mag`–`nst` (≈0,55), serta `mag`–`gap` negatif (≈−0,42). `depth`, `dmin`, dan `rms` hampir tidak berkorelasi dengan fitur lain sehingga membawa informasi yang berbeda.

## Langkah 3 — Pra-Pemrosesan Data
1. Hanya event bertipe `earthquake` yang dipertahankan (2 event bertipe `landslide` dibuang karena bukan gempa).
2. Hapus duplikasi (tidak ada).
3. **Outlier tidak dihapus.** Nilai ekstrem pada `depth` (gempa dalam hingga ±670 km) dan `mag` (gempa besar) adalah fenomena nyata, bukan kesalahan ukur; menghapusnya akan membuang justru kelompok yang paling menarik. Dampaknya diatasi dengan standarisasi.
4. Standarisasi dengan `StandardScaler` (rata-rata 0, simpangan baku 1).
5. Tidak ada fitur kategorikal yang dipakai, sehingga encoding tidak diperlukan.

In [ ]:
d = df[df["type"] == "earthquake"].drop_duplicates().reset_index(drop=True)
print("Setelah filter:", d.shape)
scaler = StandardScaler()
Z = scaler.fit_transform(d[FEATS])
Z_df = pd.DataFrame(Z, columns=[f + "_scaled" for f in FEATS])
Z_df.describe().T.round(3)

## Langkah 4 — Penentuan K (Elbow & Silhouette)

In [ ]:
ks = list(range(2, 11)); inertia = []; sil = []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(Z)
    inertia.append(km.inertia_); sil.append(silhouette_score(Z, km.labels_))
pd.DataFrame({"K": ks, "Inertia": np.round(inertia, 1), "Silhouette": np.round(sil, 4)})

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(ks, inertia, "o-", color="#1d3557"); ax[0].set(title="Elbow Method (Inertia)", xlabel="K", ylabel="Inertia")
ax[1].plot(ks, sil, "o-", color="#e63946"); ax[1].set(title="Silhouette Score", xlabel="K", ylabel="Silhouette")
plt.tight_layout(); plt.show()
print("K terbaik menurut silhouette:", ks[int(np.argmax(sil))])

## Langkah 5 — Jalankan K-Means dan evaluasi

In [ ]:
K = 3
km = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE).fit(Z)
d["cluster"] = km.labels_
Z_df["cluster"] = km.labels_
print("Inertia (SSE):", round(km.inertia_, 1))
print("Silhouette Score:", round(silhouette_score(Z, km.labels_), 3))
print(d["cluster"].value_counts().sort_index())

In [ ]:
sv = silhouette_samples(Z, km.labels_)
print(pd.Series(sv).groupby(km.labels_).mean().round(3))

**Interpretasi metrik:** inertia yang kecil berarti klaster kompak, sedangkan silhouette mendekati 1 berarti klaster terpisah baik. Silhouette ≈ 0,33 termasuk **lemah namun terlihat** (rentang 0,26–0,50): struktur klaster ada, tetapi batasnya tidak tegas, wajar untuk data seismik yang kontinu.

## Langkah 6 — Visualisasi hasil

In [ ]:
pal = sns.color_palette("Set2", K)
fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))
sns.scatterplot(data=d, x="mag", y="depth", hue="cluster", palette=pal, s=12, ax=ax[0]); ax[0].set_title("mag vs depth")
pca = PCA(2, random_state=RANDOM_STATE); P = pca.fit_transform(Z)
sns.scatterplot(x=P[:, 0], y=P[:, 1], hue=d["cluster"], palette=pal, s=12, ax=ax[1])
ax[1].set_title(f"PCA (varians {pca.explained_variance_ratio_.sum()*100:.1f}%)")
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(11, 5))
sns.scatterplot(data=d, x="longitude", y="latitude", hue="cluster", palette=pal, s=8)
plt.title("Sebaran geografis per klaster"); plt.show()

In [ ]:
fig = plt.figure(figsize=(8, 6)); ax = fig.add_subplot(111, projection="3d")
for c in range(K):
    m = d["cluster"] == c
    ax.scatter(d.loc[m, "mag"], d.loc[m, "depth"], d.loc[m, "nst"], s=6, color=pal[c], label=f"Klaster {c}")
ax.set_xlabel("mag"); ax.set_ylabel("depth"); ax.set_zlabel("nst"); ax.legend(); plt.show()

## Langkah 7 — Karakteristik tiap klaster

In [ ]:
prof_mean = d.groupby("cluster")[FEATS].mean().round(2)
prof_med = d.groupby("cluster")[FEATS].median().round(2)
display(prof_mean); display(prof_med)

In [ ]:
zm = Z_df.groupby("cluster")[[f + "_scaled" for f in FEATS]].mean(); zm.columns = FEATS
zm.T.plot(kind="bar", figsize=(10, 5), color=pal); plt.axhline(0, c="k", lw=.8)
plt.ylabel("Rata-rata z-score"); plt.title("Profil klaster"); plt.show()

In [ ]:
print(pd.crosstab(d["cluster"], d["focal_depth_regime"]))
print(pd.crosstab(d["cluster"], d["magnitude_severity_class"]))
print((d.groupby("cluster")["tsunami"].mean() * 100).round(2))

**Interpretasi klaster (K=3):**
- **Klaster 0 — gempa dangkal berkekuatan moderat (±75%)**: magnitudo rata-rata 4,67, kedalaman median 10 km, jumlah stasiun sedikit, gap besar.
- **Klaster 1 — gempa besar yang terpantau baik (±21%)**: magnitudo rata-rata 5,32, `sig` dan `nst` tinggi, `gap` kecil; memuat semua gempa kelas *major* dan *great_catastrophic* serta tingkat flag tsunami tertinggi.
- **Klaster 2 — gempa dalam (±3,7%)**: kedalaman rata-rata ≈509 km (zona *deep mantle slab*), magnitudo moderat; tidak ada flag tsunami.

## Simpan dataset siap clustering

In [ ]:
out = pd.concat([d[["id", "latitude", "longitude"]], Z_df], axis=1)
out.to_csv("Nama_Test3_ClusteringK-Means.csv", index=False)
out.head()

## Kesimpulan
K-Means dengan K=3 pada 7 fitur seismik terstandarisasi menghasilkan tiga kelompok yang bermakna secara geofisika (dangkal-moderat, besar-terpantau baik, dan dalam). Silhouette 0,331 menunjukkan struktur yang lemah tetapi terdeteksi. Pemisah paling tegas adalah kedalaman (klaster 2, silhouette 0,477); klaster 1 paling kabur (0,152).